# ML Lab · Module 6, Unit 9: Sterling volatility and VaR

**Prof. Dr. Mehmet Songur** · ML Lab, Module 6

Data: the sterling/dollar rate, daily since 1971 (FRED: DEXUSUK, the US Federal Reserve's H.10 series, public domain). The setup cell downloads a series called `fx` (date-indexed) and prints the data version (first and last date). The rate is the New York noon buying rate, not a close; the return measures the change between two consecutive available quotes.

**Copy first:** work in your own copy via **File → Save a copy in Drive**. We refer to the cells by their titles: Setup, Task A, Check A, Task B, Check B. **Have Gemini write the code**, then **audit it**.

**Rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells; read it before running.
3. If a check cell passes, it gives you a **completion code**; paste it into the box in ML Lab.

In [ ]:
#@title Setup — run it, do not change it

import base64, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

def _k(s):
    return base64.b64decode(s).decode()

def _referans(seri):
    r = 100 * np.log(seri).diff().dropna()
    gun = pd.Series(r.index, index=r.index)
    t = pd.DataFrame(index=r.index)
    for w in [5, 20, 60]:
        t[f"v{w}"] = np.sqrt((r ** 2).rolling(w).mean())
    t["y_v"] = np.sqrt((r ** 2).rolling(20).mean().shift(-20))
    t["son"] = gun.shift(-20)
    t = t.dropna()
    X, y = t[["v5", "v20", "v60"]], t["y_v"]
    e, s = t["son"] < "2015-01-01", t.index >= "2015-01-01"
    e_eski = t.index < "2015-01-01"
    cv = lambda m, XX, yy, c: cross_val_score(m, XX, yy, cv=c, scoring="r2").mean()
    out = dict(r=r, t=t, e=e, s=s,
               cv=cv(LinearRegression(), X[e], y[e], TimeSeriesSplit(5, gap=20)),
               cv_full=cv(LinearRegression(), X, y, TimeSeriesSplit(5, gap=20)),
               cv_nogap=cv(LinearRegression(), X[e], y[e], TimeSeriesSplit(5)),
               cv_rand=cv(LinearRegression(), X[e], y[e], KFold(5, shuffle=True, random_state=0)),
               test=r2_score(y[s], LinearRegression().fit(X[e], y[e]).predict(X[s])),
               test_eski=r2_score(y[s], LinearRegression().fit(X[e_eski], y[e_eski]).predict(X[s])),
               naive=r2_score(y[s], t.loc[s, "v60"]))
    v = pd.DataFrame({"y_r": r.shift(-1), "v60": np.sqrt((r ** 2).rolling(60).mean())})
    v["g"] = gun.shift(-1)
    v = v.dropna()
    ve, vs = v["g"] < "2015-01-01", v.index >= "2015-01-01"
    oran = v["y_r"] / v["v60"]
    q = np.quantile(oran[ve], 0.01)
    out.update(v=v, ve=ve, vs=vs, q=q, q_all=np.quantile(oran, 0.01), q_eski=np.quantile(oran[v.index < "2015-01-01"], 0.01),
               exc=(v.loc[vs, "y_r"] < q * v.loc[vs, "v60"]).mean(), exc_n=(v.loc[vs, "y_r"] < -2.326 * v.loc[vs, "v60"]).mean(),
               n_exc=int((v.loc[vs, "y_r"] < q * v.loc[vs, "v60"]).sum()), n_exc_n=int((v.loc[vs, "y_r"] < -2.326 * v.loc[vs, "v60"]).sum()))
    return out

fx = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/fred/DEXUSUK.csv", index_col=0, parse_dates=True).iloc[:, 0]
fx = pd.to_numeric(fx, errors="coerce").dropna()                    # dollars per pound; New York noon buying rate
print(f"Data version: fx {fx.index[0].date()} – {fx.index[-1].date()}, {len(fx)} observations (FRED updates every business day; the last digits of the results depend on the data version)")
_R = _referans(fx)

def check_A(r, table, cv_r2, test_r2, naive_r2):
    if len(r) != len(_R["r"]) or not np.allclose(np.asarray(r), _R["r"].values):
        print("❌ r differs from the expected one: r = 100 * np.log(fx).diff().dropna() (percent log return)."); return
    for c in ["v5", "v20", "v60", "y_v"]:
        if c not in table.columns:
            print(f"❌ table has no {c} column."); return
    if len(table) != len(_R["t"]) or not table.index.equals(_R["t"].index):
        print("❌ table's rows differ from the expected ones: apply dropna() after building all the columns."); return
    for c in ["v5", "v20", "v60"]:
        if not np.allclose(table[c].values, _R["t"][c].values):
            print(f"❌ {c} differs from the expected one: np.sqrt((r ** 2).rolling(w).mean()), a backward-looking window only (no center=True)."); return
    if not np.allclose(table["y_v"].values, _R["t"]["y_v"].values):
        print("❌ y_v differs from the expected one: the volatility of the next 20 observations, np.sqrt((r ** 2).rolling(20).mean().shift(-20))."); return
    t, e = _R["t"], _R["e"]
    print(f"Training: {e.sum()} decisions, last decision {t.index[e][-1].date()}, last target end {t['son'][e].iloc[-1].date()} | test: {_R['s'].sum()} decisions, first decision {t.index[_R['s']][0].date()}")
    for k, (a, b) in enumerate(TimeSeriesSplit(5, gap=20).split(t[e])):
        print(f"  CV fold {k + 1}: training last target {t['son'][e].iloc[a[-1]].date()} | validation {t.index[e][b[0]].date()} – {t.index[e][b[-1]].date()}")
    if abs(float(cv_r2) - _R["cv"]) > 1e-6:
        if abs(float(cv_r2) - _R["cv_full"]) < 1e-6:
            print("❌ cv_r2 was computed with the whole table (post-2015 included). Cross-validation for model choice is done on the training part (trn) only; the held-out test comes last."); return
        if abs(float(cv_r2) - _R["cv_rand"]) < 1e-3:
            print("❌ cv_r2 was computed with a random split (KFold, shuffle): with time series use TimeSeriesSplit(5, gap=20)."); return
        if abs(float(cv_r2) - _R["cv_nogap"]) < 1e-6 and abs(_R["cv_nogap"] - _R["cv"]) > 1e-6:
            print("❌ cv_r2 has no gap: TimeSeriesSplit(n_splits=5, gap=20)."); return
        print("❌ cv_r2 differs from the expected value: on the training rows (trn) only, LinearRegression, X = v5, v20, v60; cv=TimeSeriesSplit(n_splits=5, gap=20), scoring='r2', mean."); return
    if abs(float(test_r2) - _R["test"]) > 1e-6:
        if abs(float(test_r2) - _R["test_eski"]) < 1e-6:
            print("❌ The training cut was made by date without checking where the target ends: the targets of the last 20 decisions of 2014 reach into 2015. Use trn = table['target_end'] < '2015-01-01'."); return
        print("❌ test_r2 differs from the expected value: the model should be fitted on trn only (decisions whose target ends before 2015) and R² computed on decisions from 2015 on."); return
    if abs(float(naive_r2) - _R["naive"]) > 1e-6:
        print("❌ naive_r2 differs from the expected value: r2_score(y, table['v60']) in the test period."); return
    print(f"✅ All checks passed (training cv R² {_R['cv']:.3f}, test R² {_R['test']:.3f}, naive v60 {_R['naive']:.3f}). Your completion code: " + _k("TUw2LVZPTC00VFpD"))

def check_B(vt, q, exc_rate, exc_normal):
    v = _R["v"]
    if "y_r" not in vt.columns or "v60" not in vt.columns:
        print("❌ vt should have the columns y_r and v60 (a one-day VaR table, separate from the 20-day table)."); return
    if len(vt) != len(v) or not vt.index.equals(v.index):
        if len(vt) < len(v):
            print(f"❌ vt has {len(vt)} rows, {len(v)} expected: the last decisions may have been dropped because of unneeded columns such as y_v. Build it with y_r and v60 (and the realisation date) only, then dropna()."); return
        print("❌ vt's rows differ from the expected ones."); return
    if not np.allclose(vt["y_r"].values, v["y_r"].values):
        print("❌ vt['y_r'] should be tomorrow's return: r.shift(-1)."); return
    print(f"Calibration: last decision {v.index[_R['ve']][-1].date()}, last realisation {v['g'][_R['ve']].iloc[-1].date()} | test: {_R['vs'].sum()} decisions, {v.index[_R['vs']][0].date()} – {v.index[_R['vs']][-1].date()}, last realisation {v['g'][_R['vs']].iloc[-1].date()}")
    if abs(float(q) - _R["q"]) > 1e-9:
        if abs(float(q) - _R["q_all"]) < 1e-9:
            print("❌ The multiplier was learned from all the data (test included): use only the y_r / v60 ratio of decisions whose return was realised before 2015."); return
        if abs(float(q) - _R["q_eski"]) < 1e-9:
            print("❌ The multiplier was cut by decision date: the last decision's return belongs to 2 January 2015. Cut by realisation date: vt['realised_on'] < '2015-01-01'."); return
        print("❌ q differs from the expected value: the 1% quantile of (y_r / v60) for decisions whose return was realised before 2015, np.quantile(..., 0.01)."); return
    if abs(float(exc_rate) - _R["exc"]) > 1e-9:
        print("❌ exc_rate differs from the expected value: the mean of (y_r < q * v60) for decisions from 2015 on."); return
    if abs(float(exc_normal) - _R["exc_n"]) > 1e-9:
        print("❌ exc_normal differs from the expected value: the mean of (y_r < -2.326 * v60) for decisions from 2015 on."); return
    print(f"✅ All checks passed (multiplier {_R['q']:.4f}; exceedances {_R['n_exc']}/{_R['vs'].sum()} = {100 * _R['exc']:.2f}%, with the normal multiplier {_R['n_exc_n']}/{_R['vs'].sum()} = {100 * _R['exc_n']:.2f}%). Your completion code: " + _k("TUw2LVZBUi03QkhO"))

fx.tail(3)

## Task A — Volatility forecast

1. Daily log return: `r = 100 * np.log(fx).diff().dropna()`.
2. A DataFrame called `table`: for w = 5, 20, 60, `v{w} = np.sqrt((r ** 2).rolling(w).mean())` (past only); target `y_v = np.sqrt((r ** 2).rolling(20).mean().shift(-20))` (the next 20 available observations); the date the target ends `table["target_end"] = pd.Series(r.index, index=r.index).shift(-20)`; then `dropna()`.
3. Training and test: `trn = table["target_end"] < "2015-01-01"` (decisions whose target was fully observed before 2015), `tst = table.index >= "2015-01-01"`. The last 20 decisions of 2014 enter neither.
4. `X = table[["v5", "v20", "v60"]]`, `y = table["y_v"]`. Cross-validation for model choice **on the training part only**: the mean of `cross_val_score(LinearRegression(), X[trn], y[trn], cv=TimeSeriesSplit(n_splits=5, gap=20), scoring="r2")` is `cv_r2`.
5. Fit the model on `trn` only; R² on `tst`: `test_r2`. The R² of `table.loc[tst, "v60"]` as the naive forecast: `naive_r2`. The check cell prints the fold dates, the last training target and the first test decision.

**Audit questions:** Is there a random split (`train_test_split`, `KFold(shuffle=True)`)? Do the windows look backward? Does the target look forward? Does the last training target end before the first test decision?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(r, table, cv_r2, test_r2, naive_r2)

## Task B — Value at risk (1% VaR)

1. Build a table for VaR **separate** from the 20-day table (otherwise the missing values of `y_v` drop the last 19 decisions): `vt = pd.DataFrame({"y_r": r.shift(-1), "v60": np.sqrt((r ** 2).rolling(60).mean())})`, the date the return is realised `vt["realised_on"] = pd.Series(r.index, index=r.index).shift(-1)`, then `dropna()`.
2. Multiplier: the 1% quantile of `y_r / v60` for decisions **whose return was realised before 2015** (`vt["realised_on"] < "2015-01-01"`): `q = np.quantile(..., 0.01)`. This is the multiplier of the 1% lower return threshold (negative); VaR in loss terms = −q × v60.
3. Exceedance: `vt["y_r"] < q * vt["v60"]`. Its mean over decisions from 2015 on: `exc_rate`.
4. Do the same with the normal multiplier (−2.326): `exc_normal`. Compare both with the 1% target; also print the number of exceedances, n and how exceedances are spread across years.

**Audit questions:** Was the multiplier learned from the training period only? Is the last return in the calibration before 2015? In which years do exceedances pile up?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(vt, q, exc_rate, exc_normal)

## Your audit notes

- Did Gemini work in a time-respecting way? What did you need to fix?
- How do the sterling results compare with euro/dollar? (cv_r2, test_r2, naive_r2)
- How would you explain the VaR results to a risk committee? (exceedance rate, gap with the normal multiplier, limitations)